# AML - LEAVE ONE VARIANT OUT - PLOT 

In [ ]:
library(data.table)
library(tidyverse)
library(readr)
library(ggplot2)
library(forcats)

In [ ]:
data <- fread("EUR_aml_phenotype_deprivation_index_leave_one_variant_out.txt")

In [ ]:
comb_colors <- c(
  "Match : AlphaMissense-0.564" = paletteer::paletteer_d("LaCroixColoR::Lemon")[1],
  "Match : LOFTEE-HC" = paletteer::paletteer_d("LaCroixColoR::Lemon")[2],
  "Match : SpliceAI-0.8" = paletteer::paletteer_d("LaCroixColoR::Lemon")[3],
  "No Match : AlphaMissense-0.564" = paletteer::paletteer_d("LaCroixColoR::Lemon")[4],
  "No Match : LOFTEE-HC" = paletteer::paletteer_d("LaCroixColoR::Lemon")[5],
  "No Match : SpliceAI-0.8" = paletteer::paletteer_d("LaCroixColoR::Lemon")[6]
)

In [ ]:
df <- data.frame(
  GENE = c("ENSG00000119772", "ENSG00000168769", "ENSG00000161547", "ENSG00000182054"),
  SYMBOL = c('DNMT3A', 'TET2', 'SRSF2', 'IDH2'), 
  P = c(7.782313e-10, 3.346137e-09, 1.511365e-07, 1.338055e-06)
)

In [ ]:
for (gene in unique(data$Region)){ 
print("=================================================================================")
print(paste("Print top variants for gene :", gene))
print("=================================================================================")
top <- 
  data %>%
  filter(anno != "Synonymous") %>% 
  filter(Group == "Cauchy") %>% 
  mutate(var = as.character(var)) %>% 
  filter(Region == gene) %>% 
  inner_join(df, by = c("Region" = "GENE")) %>% 
  mutate(LOVO_Less_Significant_Pvalue = ifelse(Pvalue < P, TRUE, FALSE)) %>% 
  mutate(LOVO_Non_Significant_Pvalue = ifelse(Pvalue < (0.05/18064), TRUE, FALSE)) %>% 
  dplyr::select(var, SYMBOL, anno, LOVO_Less_Significant_Pvalue, LOVO_Non_Significant_Pvalue, 
                Region, Pvalue, Clinvar_Match, GDC_Match, SYMBOL, P) %>% 
  rename('Variant' = "var", 
         'Symbol' = "SYMBOL", 
         'Annotation' = 'anno',
         'LOVO_Pvalue' = 'Pvalue', 
         'Original_PValue' = 'P') %>% 
  arrange(desc(LOVO_Pvalue))
    
  print(head(top))

}


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 20)

In [ ]:
for (gene in unique(data$Region)){ 

plot <- 
  data %>%
  filter(anno != "Synonymous") %>% 
  filter(Group == "Cauchy") %>% 
  mutate(var = as.character(var)) %>% 
  filter(Region == gene) %>% 
  mutate(
    Negative_Log_Pvalue = -log10(Pvalue), 
    var = fct_reorder(var, - Negative_Log_Pvalue)
  ) %>%
  arrange(Negative_Log_Pvalue) %>% 
  head(n = 25) %>% 
  mutate(color_status = paste0(Match_Status," : ",anno)) %>% 
  ggplot(aes(y = var, 
             x = Negative_Log_Pvalue)) +
  geom_segment(aes(
    y = var,
    yend = var,
    x = 0,
    xend = Negative_Log_Pvalue
  )) +
  geom_bar(aes(
    y = var,
    x = Negative_Log_Pvalue,
    fill = color_status), color = 'black', stat = "identity") +
  geom_vline(xintercept = -log10(0.05/18064), 
            linetype = "dashed", 
            linewidth = 2,
            color = "red") +
    geom_vline(xintercept = filter(df, GENE == gene) %>% 
                            mutate(Negative_Log_Pvalue = -log10(P)) 
                             %>% pull(Negative_Log_Pvalue), 
                             linetype = "dashed", linewidth = 2) +
  theme_minimal() +
  theme(
    axis.title.y = element_blank(),
    plot.title = element_text(size = 16)
  ) + 
  labs(fill = "Annotation",
       x = "-log10(p-values)",
       title = paste("Gene:", filter(df, GENE == gene) %>% pull(SYMBOL))) + 
  scale_fill_manual(values = comb_colors)

  print(plot)

}
